In [ ]:
# system utilities
import os
import glob

# modules to query skymapper 
import urllib.request

# plotting module
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# numeric and table data modules
import numpy as np
from numpy import errstate,isneginf,array

# dataframes module
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# module to read Fits files
from astropy.io import fits

# image visualization and World Coordinate System modules
from astropy.visualization import simple_norm
from astropy.wcs import WCS

# units and coordinate handling
from astropy import units as u
from astropy.coordinates import SkyCoord

# Apperture photometry module
from photutils import SkyCircularAperture, aperture_photometry




In [ ]:
def getKeyListVal(filename, keys):
    keys_dict = {}
    with fits.open(filename) as hdu_list:
        primary_header = hdu_list[0].header
        for key in keys:
            keys_dict[key] = primary_header[key]
        return keys_dict


# Only skymapper defined and only for r_SDSS and NB_659
# but it should be easily extensible
#
class ZPCalibration:    
    """A class to contain a pandas dataframe with the data
    to calibrate frame and the zero points needed for the calibration."""
    def __init__(self, frame, calibrator, mlim, mag_min, mag_max, dmag_filter):
        self.frame = frame
        self.calibrator = calibrator
        self.mlim = mlim
        self.mag_min =  mag_min
        self.mag_max = mag_max
        self.dmag_filter = dmag_filter
        self.band = getKeyListVal(frame, ['FILTER'])['FILTER'].strip() 

        # we load the df with the calibrator data
        # we need positions and magnitudes
        #
        if calibrator == 'skymapper':
            self.df = self.getSkyMapperDf()
        else:
            self.df = None
            
        # Now we load the dataframe with the raw magnitudes in band self.band
        # Note that self.band will be the name of the column of the data frame
        
        # we add the raw magnitudes to self.df
        self.doPhot()
        
        # here we get the zero point
        self.zp, self.slope, self.sigma = self.getZP()
        
        # we get the conversion factor to Flambda in erg/s/cm^2
        self.zp2 = self.getFlambdaZP(self.getPivotWavelength(self.band))
        # print('zp1: ', self.zp, 'zp2: ', self.zp2, '(zp2+zp1)/2.5: ', (self.zp2+self.zp)/2.5)
        self.convFactor_ADU_Flambda = 10**((self.zp2 + self.zp)/2.5)
        self.filterWidth = self.getFilterWidth(self.band)
        # print(self.convFactor_ADU_Flambda)
        
        # here we add std magnitudes to self.df
        self.getStdMagsFromFrame()
            
    def getSkyMapperDf(self):
        coord = getKeyListVal(self.frame, ['CRVAL1', 'CRVAL2'])
        ra_str = ("%-10.5f" % coord['CRVAL1']).strip()
        de_str = ("%-10.5f" % coord['CRVAL2']).strip()
        radius = '0.7'
        self.query_str = "http://skymapper.anu.edu.au/sm-cone/public/query?RA="+ra_str+"&DEC="+de_str+"&SR=0.80&VERB=3&RESPONSEFORMAT=CSV"
        print(self.query_str)
        with urllib.request.urlopen(self.query_str) as f:
            charset = f.headers.get_content_charset()
            # print(charset)
            df_cat = pd.DataFrame([x.split(',') for x in f.read().decode(charset).split('\r\n')])

        # we set column 1 as the header collumn
        df_cat.columns = df_cat.iloc[0]

        # we select only relevant collumns -- FILTER DEPENDENT
        if self.band == 'r_SDSS' or self.band == 'NB_659':
            df_band_columns = ['g_psf', 'e_g_psf', 'r_psf', 'e_r_psf']
            df_pos_columns = ['raj2000', 'dej2000']
            df_total_columns = ['object_id']+df_pos_columns+['class_star'] + df_band_columns
            df_cat = df_cat[df_total_columns][1:]
            
            # update Class accordingly
            self.ref_band_name = 'sm_r_SDSS'
            self.ref_sideband_name = 'sm_g_SDSS'

        # we replace empty values by nan
        for column in df_pos_columns+df_band_columns:
            df_cat[column]=df_cat[column].replace('',np.nan).astype(float)
            df_cat[column] = df_cat[column].astype('float64')        

        # we select according to mlim and columns with valid coords
        bool_select = (df_cat['r_psf'] < mlim) & \
            (df_cat['g_psf'] < mlim) & \
            (df_cat['raj2000'] == df_cat['raj2000']) & \
            (df_cat['dej2000'] == df_cat['dej2000'])
        df_cat = df_cat[bool_select]   

        # we rename collumns to have a uniform interface for all calibrators
        df_cat.rename(columns={'raj2000': 'ra', 'dej2000': 'dec', \
                     'r_psf': 'ref_band', 'e_r_psf': 'ref_band_e', \
                    'g_psf': 'ref_sideband', 'e_g_psf': 'ref_sideband_e'}, inplace=True)

    
        return df_cat
        

    def doPhot(self):
        ra = np.array(self.df['ra'])*u.deg
        dec = np.array(self.df['dec'])*u.deg

        # Define positions and apertures
        positions = SkyCoord(ra=ra, dec=dec,frame='icrs')
        self.aperture = SkyCircularAperture(positions, r=2. * u.arcsec)

        # open image and read
        self.data, self.hdu = fits.getdata(self.frame, header=True)
        # abd we plot
        # read wcs from image keywords
        self.wcs = WCS(self.frame)  

        # we do the aperture photometry -- first good enough version.
        # For a better approach we should do curve of growth..
        #
        phot_table = aperture_photometry(self.data, self.aperture, wcs=self.wcs)
        phot_table['aperture_sum'].info.format = '%.8g'  # for consistent table output

        # we read the magnitudes from the tables ocam_band
        # df1 = df1.assign(e=pd.Series(np.random.randn(sLength)).values)
        self.df = self.df.assign(raw_mags=pd.Series(-2.5*np.log10(np.array(phot_table['aperture_sum']))).values)
        # display(self.df)
        
    
    # getZP(ref_band, ref_sideband, ocam_band, mag_min, mag_max, dmag_filter)
    

    def getZP(self):
        
        ref_band = np.array(self.df['ref_band'])
        ref_sideband = np.array(self.df['ref_sideband'])
        ocam_band = np.array(self.df['raw_mags'])
        
        X = []
        Xg = []
        Y = []
        for i in range(len(ref_band)):
            if (ref_band[i] == ref_band[i]) and \
                not (np.isinf(ocam_band[i]) or np.isnan(ocam_band[i])) and \
                (ref_band[i] > self.mag_min and ref_band[i] < self.mag_max):
                X.append(ref_band[i])
                Xg.append(ref_sideband[i])
                Y.append(ocam_band[i])

        X = np.array(X)
        Xg = np.array(Xg)
        Y = np.array(Y)
        
        # needs to modify this to force slope=1.00
        
        from scipy.optimize import fsolve
        s = 1.0000
        c = fsolve(lambda i: ((Y - (s*X + i))**2).sum(), 0.) # ((Y - (s*X + i))**2).sum()
        model = (s, c[0])

        # we do one iteration of sigma clipping
        X1 = []
        Xg1 = []
        Y1 = []
        for i in range(len(X)):
            ypred = model[0]*X[i] + model[1]
            if np.abs(Y[i] - ypred) < self.dmag_filter:
                X1.append(X[i])
                Xg1.append(Xg[i])
                Y1.append(Y[i])

        X1 = np.array(X1)
        Xg1 = np.array(Xg1)
        Y1 = np.array(Y1)

        # we fix slope to be 1.000 need to add colour term...
        s = 1.0000
        c = fsolve(lambda i: ((Y1 - (s*X1 + i))**2).sum(), c[0]) # ((Y - (s*X + i))**2).sum()
        model = (s, c[0])
        
        print(model)
        #model = np.polyfit(X1, Y1, 1)
    
        # with respect to this model
        ocam_prediced_mag = []
        for i in range(len(ref_band)):
            ocam_prediced_mag = np.append(ocam_prediced_mag, model[0]*ref_band[i] + model[1])
    
        # filter nan and inf values
        good_values1 = ~np.isnan(ocam_band)
        good_values2 = ~np.isinf(ocam_band)
        good_values = good_values1 & good_values2
    
        sigma = np.std(ocam_band[good_values] - ocam_prediced_mag[good_values])
    
        slope = model[0]
        zp = model[1]
        return zp, slope, sigma
    
    def getFlambdaZP(self, lam):
        zp = -2.5*np.log10(3.34*10**(-19)) -5.0*np.log10(lam) - 48.60
        return zp

    # Girardi 2004: https://www.aanda.org/articles/aa/full/2004/28/aa0250-04/aa0250-04.html
    # SSDS DR1: http://classic.sdss.org/dr1/algorithms/fluxcal.html
    # 

    def getFilterWidth(self, band):
        pivWave = {'u_SDSS': 560.0,
              'g_SDSS': 1340.0,
              'r_SDSS': 1350.0,
              'NB_659':  105.0, # This is the VPHAS+ filter
              'i_SDSS': 1550.0,
              'z_SDSS':  800.0,
              'B_JOHN': 1030.0, # This is the Bessell B isophotal mag
              'V_JOHN':  980.0} # This is the Bessel V isophotal mag
    
        if band in pivWave.keys():
            return pivWave[band]
        else:
            raise Exception('getFilterWidth: unrecognize filter pass band %s'%(band))        
        
    def getAB_delta(self, band):
        abDelta = {'u_SDSS': 0.0,
              'g_SDSS': 0.0,
              'r_SDSS': 0.0,
              'NB_659': 0.0,
              'i_SDSS': 0.0,
              'z_SDSS': 0.0,
              'B_JOHN': 0.04, # This is the Bessell B isophotal mag
              'V_JOHN': 0.0} # This is the Bessel V isophotal mag
    
        if band in abDelta.keys():
            return abDelta[band]
        else:
            raise Exception('getAB_delta: unrecognize filter pass band %s'%(band))
    
    def getPivotWavelength(self, band):
        """Returns the pivot wavelength needed in the
        conversion from AB magnitudes to physical flux."""
        pivWave = {'u_SDSS': 3557.0,
              'g_SDSS': 4702.0,
              'r_SDSS': 6175.0,
              'NB_659': 6590.0,
              'i_SDSS': 7491.0,
              'z_SDSS': 8946.0,
              'B_JOHN': 4371.0, # This is the Bessell B isophotal mag
              'V_JOHN': 5478.0} # This is the Bessel V isophotal mag
    
        if band in pivWave.keys():
            return pivWave[band]
        else:
            raise Exception('getPivotWavelength: unrecognize filter pass band %s'%(band))
    
    # Get the zero point when DN are in physical unitsç
    # of erg/s/cm^-2/Angstrom
    def getPhysicalUnitsZP(self):
        """returns the zero point to add to -2.5log(flux)
        to get the AB magnitudes in band self.band"""
        return -2.5*np.log10(3.34*10**(-19)) - 5.0*np.log10(self.getPivotWavelength(self.band)) - 48.60
 
    
    def getStdMagsFromFrame(self):
        #data, hdu = fits.getdata(self.frame, header=True)
        #wcs = WCS(self.frame)  
        #data = data*self.convFactor_ADU_Flambda
        phot_table = aperture_photometry(self.data*self.convFactor_ADU_Flambda, self.aperture, wcs=self.wcs)
        phot_table['aperture_sum'].info.format = '%.8g'  # for consistent table output
        ocam_band = -2.5*np.log10(np.array(phot_table['aperture_sum'])) + self.getPhysicalUnitsZP()

        # we add column with standard magnitude to self.df
        self.df = self.df.assign(std_mags=pd.Series(ocam_band).values)
        # display(self.df)

        

In [ ]:
print('current directory:', os.getcwd())
current_dir = os.getcwd()
os.chdir(current_dir)

reduced_dir = '/home/fselman/omegacam/fernando/reduced/science/J1649-3307_164955p600_m330702p0/r_SDSS/2017-04-13T05:11:38/'
current_dir = '/home/fselman/omegacam/fernando/'

frame = reduced_dir+'coadd.fits'

mlim = 18.0
mag_min = 13.5
mag_max = 17.0
dmag_filter = 0.2

zp_cal = ZPCalibration(frame, 'skymapper', mlim, mag_min, mag_max, dmag_filter)
# print(zp_cal.ref_band_name)

print('zp_cal.convFactor_ADU_Flambda: ', zp_cal.convFactor_ADU_Flambda)


In [ ]:
# we plot the magnitudes from the photometry table
ref_band = np.array(zp_cal.df['ref_band'])
ref_sideband = np.array(zp_cal.df['ref_sideband'])
ocam_band = np.array(zp_cal.df['std_mags'])

data = zp_cal.data
wcs = zp_cal.wcs

sigma = zp_cal.sigma
mag_max = zp_cal.mag_max
mag_min = zp_cal.mag_min

# we plot the ocam vs skymapper relation
# figsize=(8, 4)
fig = plt.figure(figsize=(14.26, 11.0))

gs = gridspec.GridSpec(nrows=4, ncols=5, wspace=0.0) # width_ratios=[1,1,1,1,1])
ax1 = fig.add_subplot(gs[:-2,:-3])
ax2 = fig.add_subplot(gs[-2, :-3], sharex=ax1)
ax3 = fig.add_subplot(gs[:-1,  -3:])
ax4 = fig.add_subplot(gs[-1,:])


#plt.figure(figsize = (10,10))
#ax1.set_aspect('equal')
ax1.scatter(ref_band, ocam_band,  color='black', s=2)
ax1.set_xlim(8.0, mag_max+1.0)
ax1.set_ylim(8.0, mag_max+1.0)

# delta zp vs mag
ax2.scatter(ref_band, ocam_band - ref_band,  color='black', s=2)
ax2.set_ylim(-10.0*sigma, 10.0*sigma)
ax2.axhline(0.0, ls=':')
ax2.axvline(mag_min, ls=':')
ax2.axvline(mag_max, ls=':')

# and we plot apertures on image
  
norm = simple_norm(data, 'sqrt', percent=99)
ax3.imshow(data, norm=norm)
zp_cal.aperture.to_pixel(wcs).plot(color='white', lw=1)

plt.setp(ax3.get_xticklabels(), visible=False)
plt.setp(ax3.get_yticklabels(), visible=False)

# delta zp vs color
color = ref_sideband - ref_band
select = ref_band > mag_min

# the slope of 0.994 gives a bias at fainter magnitudes
# that translate into an offset. If we set slope to 1
# for this plot we get an offset 0.1 mags.
#
ax4.scatter(color[select], ocam_band[select] - ref_band[select],  color='black', s=2)
ax4.set_ylim(-3.0*sigma, 3.0*sigma)
ax4.set_xlim(-0.5, 2.5)
ax4.axhline(0.0, ls=':')
#plt.setp([ax1, ax2], title='ZP Conversion')

#plt.ylim(7, mlim)
plt.show()
#print(color)
#print(phot_table)

In [ ]:
# Plot zeropoint versus magnitude
x = ref_band
xg = ref_sideband
y = ocam_band

plt.figure(figsize = (10,3))
#plt.xlim(8.5, 17.0)
plt.ylim(-0.2, 0.2)
plt.scatter(xg - x, y - x,  color='black', s=2)
plt.title('OCAM ZP')
plt.xlabel('SkyMapper (g-r)_SDSS')
plt.ylabel('OCAM - SkyMapper r_SDSS')
plt.axhline(y=0.0)
#plt.xticks(())
#plt.yticks(())

#plt.plot(X1, Y1 - (model[0]*X1+model[1]), color='red',linewidth=2)
plt.show()

# Plot zeropoint versus color
plt.figure(figsize = (10,3))
plt.xlim(8.5, 17.0)
plt.ylim(-0.2, 0.2)
plt.scatter(x, y - x,  color='black', s=2)
plt.title('OCAM ZP')
plt.xlabel('SkyMapper r_SDSS')
plt.ylabel('OCAM - SkyMapper r_SDSS')
plt.axhline(y=0.0)
#plt.xticks(())
#plt.yticks(())

plt.show()

# y = model[0]*x + model[1]
print(zp_cal.slope, zp_cal.zp)
mag_zeropoint = -zp_cal.zp
print('mag_zeropoint: %6.3f'%(mag_zeropoint))
#fits.setval(reduced_dir+'coadd.fits', 'MAGZP', value=mag_zeropoint, comment='Magnitude zp from SkyMapper', before='EQUINOX', after=None, savecomment=False)

# read wcs info to get plate scale
wcs_keys = getKeyListVal(reduced_dir+'coadd.fits', ['CD1_1', 'CD1_2', 'CD2_1', 'CD2_2'])
cd1_1 = wcs_keys['CD1_1']
cd1_2 = wcs_keys['CD1_2']
cd2_1 = wcs_keys['CD2_1']
cd2_2 = wcs_keys['CD2_2']
plate_scale = np.sqrt(-(cd1_1*cd2_2 - cd1_2*cd2_1))*3600.0
print('plate_scale: ', plate_scale)

# now we do some statistics to get the sky
from numpy import errstate,isneginf,array
sky_median = np.nanmedian(zp_cal.data)*zp_cal.convFactor_ADU_Flambda
sky_flux_std = np.nanstd(zp_cal.data)*zp_cal.convFactor_ADU_Flambda

# we calculate the background std. deviation sigma-clipping once
sky_flux_std = np.std(zp_cal.data[zp_cal.data*zp_cal.convFactor_ADU_Flambda < sky_median+5.0*sky_flux_std]*zp_cal.convFactor_ADU_Flambda)
print('sky_flux_standard-deviation: ', sky_flux_std)


In [ ]:
print('sky_flux_standard-deviation: ', sky_flux_std)

In [ ]:
# now we do some statistics to get the sky
# But we should consider that during the
# swarp process we lost the sky. Therefore,
# we will retrieve the sky from 
# the OMEGA.YYYY-MM-DDThh:mm:ss.sss_pre_wcs.fits files
# We loop through every file
# retrieve the median sky values from each chip
# create the median and std deviation per frame
# add these keywords to the final frame
# at the end we add the mean over all medians
# to the coadd.fits file.
ext_name_list = ["ESO_CCD_#%2d"%(i) for i in range(65,97)]
  

list_to_process = glob.glob(reduced_dir+'OMEGA.*_pre/OMEGA.*_pre_wcs.fits')

sky_mean_list = []
sky_std_list = []
if len(list_to_process) > 0:
    i = 0
    for file_name in list_to_process:
        i += 1
        if i <= len(list_to_process):
            hdu_list = fits.open(file_name)
            print('Finding sky %i/%i: %s'%(i, len(list_to_process), os.path.basename(file_name),))
            median_list = []
            for extname in ext_name_list:
                hdr = fits.getheader(file_name, extname=extname)
                data = fits.getdata(file_name, extname=extname)
                median_list.append(np.nanmedian(data))
            median_list = np.array(median_list)
            
            sky_mean = np.nanmean(median_list)
            sky_mean_list.append(sky_mean)
            
            sky_std = np.nanstd(median_list)            
            sky_std_list.append(sky_std)
            print(os.path.basename(file_name), sky_mean, sky_std,)
            
            #hdu_list.info()
print('Done.')



In [ ]:
# Now we determine the overall sky level
# to be added to coadd.fits
#
# Actually, we need the convertion factor
# we need to multiply the pixel values so that
# we get a signal in erg/s/cm^-2/Angstrom
# that will also permit us to get magnitudes
# with aperture photometry with the formula
#
# mag = f0 - 2.5 log10(Sum_pix_ij val_ij)
# 
# where f0 is given in the header keyword
#
# HIERARCH MEFPP QC F0_PHYSICAL
#
sky_flux = 'HIERARCH MEFPP QC SKY FLUX'
sky_mag = 'HIERARCH MEFPP QC SKY MAG'
nvalid_skies = len(sky_mean_list)

print(nvalid_skies)
good_sky = (nvalid_skies == len(list_to_process))
print('Good sky?: ', good_sky)
sky_mean_mean = np.nanmean(sky_mean_list)
sky_mean_max = np.nanmax(sky_mean_list)
sky_mean_min = np.nanmax(sky_mean_list)

std_mean_max = np.nanmax(sky_std_list)
std_mean_min = np.nanmin(sky_std_list)

sky_delta_mag = 2.5*sky_flux_std/(sky_mean_mean*np.log(10.0))
print('sky_mag: ', -2.5*np.log10(sky_mean_mean/plate_scale**2) + mag_zeropoint)
print('sky_delta_mag: ', sky_delta_mag)



# Could not make this work. To continue testing change the cell type from Markdown to Code

from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.vo_conesearch import conesearch
from astroquery.vo_conesearch.vos_catalog import VOSDatabase
import warnings

## To explore catalog availability uncomment this file and then go to the cell named explore catalogues...

from astroquery.vo_conesearch.validator import conf as validator_conf
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    registry_db = VOSDatabase.from_registry(
        validator_conf.conesearch_master_list, encoding='binary',
        cache=False)
len(registry_db)    
conesearch.list_catalogs()
## EXPLORE CATALOGS

### the commented lines were used to explore to find the skymapper catalog
#sm_a2_list = registry_db.list_catalogs(pattern='skymapper')
#sm_a2_list
sm_catalog = registry_db.get_catalog('SkyMapper Southern Sky Survey. DR1.1 (Wolf+, 2018) 1')
print(sm_catalog)
#sm_catname = 'SkyMapper Southern Sky Survey. DR1.1 (Wolf+, 2018) 1'
#skymapper_url = 'http://vizier.u-strasbg.fr/viz-bin/conesearch/II/358/smss?'
#sm = registry_db.get_catalog_by_url(skymapper_url)
print(sm)

c = SkyCoord.from_name('CP Pupis')
c
sm_catname = 'SkyMapper Southern Sky Survey. DR1.1 (Wolf+, 2018) 1'
result = conesearch.conesearch(c, 0.1 * u.degree, catalog_db=sm_catname)